# 02 · Model outputs (after training / inference)

For one experiment run: metrics summary, training curves, **input vs. output** on sample test images,
and the worst failure cases.

Prerequisite: `python scripts/run_experiment.py --config configs/experiments/<name>.yaml`
(sample images work as soon as `train/weights/best.pt` exists).

In [ ]:
import os
import sys
from pathlib import Path

# Run from the repo root so config paths resolve the same way as in scripts/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.dpi"] = 110

In [ ]:
from omegaconf import OmegaConf
from ultralytics import YOLO

from src.evaluation import load_gt_masks, predict_instances
from src.targeting.metrics import evaluate_image
from src.utils.io import load_json, read_jsonl
from src.visualization import GT_COLOR, PRED_COLOR, draw_instances, load_rgb

RUN_DIR = Path("experiments/baseline_yolo11n_seg")
NUM_SAMPLES = 6
SEED = 0

cfg = OmegaConf.load(RUN_DIR / "config.yaml")          # config snapshot of that run
prepared_dir = Path(cfg.dataset.prepared_dir)
split, methods = cfg.eval.split, list(cfg.targeting.methods)
class_names = load_json(prepared_dir / "prepare_info.json")["class_names"]
model = YOLO(str(RUN_DIR / "train" / "weights" / "best.pt"))
results = load_json(RUN_DIR / "results.json") if (RUN_DIR / "results.json").exists() else None
print("primary targeting method:", methods[0], "| conf:", cfg.eval.conf)

## Metrics summary (`results.json`)

In [ ]:
if results:
    seg = pd.DataFrame({k: results["segmentation"][k] for k in ("box", "mask")}).T
    display(seg.round(3))
    tgt = pd.DataFrame({m: {"shot_precision": v["shot_precision"], "weed_hit_rate": v["weed_hit_rate"],
                            "shot_f1": v["shot_f1"], "point_error_px (median)": v["point_error_px"]["median"],
                            "point_error_rel (median)": v["point_error_rel"]["median"],
                            "extraction ms/img": v["extraction_ms_per_image"]}
                        for m, v in results["targeting"]["methods"].items()}).T
    display(tgt.round(3))
    print(f"inference {results['targeting']['inference_ms_per_image']:.1f} ms/img, "
          f"{results['model']['num_parameters'] / 1e6:.2f} M params, {results['model']['gflops']} GFLOPs")
else:
    print("results.json not written yet (run not finished)")

## Training curves

In [ ]:
history = pd.read_csv(RUN_DIR / "train" / "results.csv")
history.columns = history.columns.str.strip()
loss_cols = [c for c in history.columns if c.endswith("_loss")]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
history.plot(x="epoch", y=loss_cols, ax=axes[0], title="losses")
history.plot(x="epoch", y=[c for c in history.columns if c.startswith("metrics/mAP")], ax=axes[1], title="val mAP")
plt.tight_layout(); plt.show()

## Input → output on sample test images
Left: model input. Middle: ground truth (red mask + GT target point).
Right: prediction (green masks with class/confidence + laser target points of the primary method).

In [ ]:
def show_prediction(image_path, method=methods[0]):
    image_bgr = cv2.imread(str(image_path))
    h, w = image_bgr.shape[:2]
    gt_masks, _ = load_gt_masks(prepared_dir / "labels" / split / f"{image_path.stem}.txt", h, w)
    pred = predict_instances(model, image_bgr, conf=cfg.eval.conf, iou=cfg.eval.iou, imgsz=cfg.eval.imgsz)
    points = evaluate_image(pred.masks, gt_masks, [method], cfg.eval.match_iou)["methods"][method]

    image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    labels = [f"{class_names[c]} {s:.2f}" for c, s in zip(pred.class_ids, pred.confidences)]
    panels = (image,
              draw_instances(image, gt_masks, points["gt_points"], color=GT_COLOR),
              draw_instances(image, pred.masks, points["pred_points"], labels, color=PRED_COLOR))
    titles = ("input", f"ground truth ({len(gt_masks)} weeds)",
              f"prediction ({len(pred.masks)} shots, {points['shots_on_weed']} on a weed)")
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    for ax, panel, title in zip(axes, panels, titles):
        ax.imshow(panel); ax.set_title(title); ax.axis("off")
    fig.suptitle(f"{image_path.name} | {method} | {pred.inference_ms:.0f} ms"); plt.tight_layout(); plt.show()


images = sorted((prepared_dir / "images" / split).glob("*"))
rng = np.random.default_rng(SEED)
for image_path in rng.choice(images, size=min(NUM_SAMPLES, len(images)), replace=False):
    show_prediction(image_path)

## Worst cases (from `eval/predictions.jsonl`)
Ranked by missed weeds + shots that did not land on a weed (primary method).

In [ ]:
records = read_jsonl(RUN_DIR / "eval" / "predictions.jsonl")
per_image = pd.DataFrame([{
    "image_id": r["image_id"], "image_path": r["image_path"], "num_gt": r["num_gt"], "num_pred": r["num_pred"],
    "missed_weeds": r["num_gt"] - r["methods"][methods[0]]["weeds_hit"],
    "false_shots": r["methods"][methods[0]]["shots"] - r["methods"][methods[0]]["shots_on_weed"],
} for r in records])
per_image["errors"] = per_image.missed_weeds + per_image.false_shots
worst = per_image.sort_values("errors", ascending=False).head(3)
display(worst)
for path in worst.image_path:
    show_prediction(Path(path))

## Point error distribution per method
Distance between predicted and GT target point for matched weeds, relative to the weed's equivalent
radius sqrt(area/π); below 1 ≈ inside the weed.

In [ ]:
errors = {m: [e for r in records for e in r["methods"][m]["point_errors_rel"]] for m in methods}
plt.figure(figsize=(7, 4))
plt.boxplot([errors[m] for m in methods], showfliers=False)
plt.xticks(range(1, len(methods) + 1), methods)
plt.axhline(1.0, color="grey", ls="--", lw=1); plt.ylabel("point error / weed radius")
plt.tight_layout(); plt.show()